# lyp-sync — Lip Sync from Image + Audio

Generate a talking-head video from a **portrait image** and **speech audio** (MP3 or WAV).

Powered by [SadTalker](https://github.com/OpenTalker/SadTalker) on Google Colab.

---

### Before you start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells **top to bottom** (Setup once per session, then Upload + Generate)
3. A ~60s clip takes about **10–30 min** on a free T4 GPU

In [ ]:
# @title 1. Check GPU
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU ready: {name} ({vram:.1f} GB)")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run this cell."
    )

In [ ]:
# @title 2. Install SadTalker (~5 min first run)
import os
import subprocess
from pathlib import Path

ROOT = Path("/content/lyp-sync")
SADTALKER = ROOT / "SadTalker"
CHECKPOINTS = ROOT / "checkpoints"
UPLOADS = ROOT / "uploads"
OUTPUTS = ROOT / "outputs"

for d in (ROOT, CHECKPOINTS, UPLOADS, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

if not (SADTALKER / "inference.py").exists():
    !git clone --depth 1 https://github.com/OpenTalker/SadTalker.git {SADTALKER}
else:
    print("SadTalker already cloned.")

!apt-get -qq install -y ffmpeg
!pip install -q torch torchvision torchaudio
!pip install -q -r {SADTALKER}/requirements.txt
!pip install -q 'imageio>=2.31' 'imageio-ffmpeg>=0.4.9' ipywidgets

# Patch basicsr for newer torchvision
import basicsr
degradations = Path(basicsr.__file__).parent / "data/degradations.py"
text = degradations.read_text()
old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
new = """try:
    from torchvision.transforms.functional_tensor import rgb_to_grayscale
except ImportError:
    from torchvision.transforms.functional import rgb_to_grayscale"""
if old in text:
    degradations.write_text(text.replace(old, new, 1))
    print("Patched basicsr.")

print("Install complete.")

In [ ]:
# @title 3. Download model checkpoints (~1.5 GB, once per session)
RELEASE = "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc"
files = [
    "SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors",
    "mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar",
]

for name in files:
    dest = CHECKPOINTS / name
    if dest.exists():
        print(f"skip: {name}")
        continue
    print(f"download: {name}")
    !wget -q --show-progress -O {dest} {RELEASE}/{name}

print("Checkpoints ready.")

In [ ]:
# @title 4. Upload image + audio
from google.colab import files

def _save_upload(label):
    print(label)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    name = next(iter(uploaded))
    dest = UPLOADS / name
    dest.write_bytes(uploaded[name])
    print(f"  → {dest}")
    return dest

image_path = _save_upload("Upload your portrait image (JPG/PNG)...")
audio_path = _save_upload("\nUpload your speech audio (MP3/WAV)...")

In [ ]:
# @title 5. Settings
size = 256  # @param [256, 512] {type:"raw"}
preprocess = "crop"  # @param ["crop", "resize", "full", "extcrop", "extfull"]
still_mode = False  # @param {type:"boolean"}
use_enhancer = False  # @param {type:"boolean"}
batch_size = 2  # @param {type:"slider", min:1, max:4, step:1}
output_name = "lipsync_output.mp4"  # @param {type:"string"}

In [ ]:
# @title 6. Generate lip-sync video
import subprocess
import sys
from IPython.display import Video, display

cmd = [
    sys.executable, "-u", str(SADTALKER / "inference.py"),
    "--driven_audio", str(audio_path),
    "--source_image", str(image_path),
    "--checkpoint_dir", str(CHECKPOINTS),
    "--result_dir", str(OUTPUTS),
    "--size", str(size),
    "--preprocess", preprocess,
    "--batch_size", str(batch_size),
]
if still_mode:
    cmd.append("--still")
if use_enhancer:
    cmd.extend(["--enhancer", "gfpgan"])

print("Running SadTalker...")
print(" ".join(cmd))
print("-" * 60)

env = os.environ.copy()
env["PYTHONPATH"] = str(SADTALKER)
proc = subprocess.run(cmd, cwd=str(SADTALKER), env=env)
if proc.returncode != 0:
    raise RuntimeError(f"SadTalker failed with exit code {proc.returncode}")

results = sorted(OUTPUTS.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not results:
    raise RuntimeError("No output video found in outputs/")

final = OUTPUTS / output_name
shutil.copy2(results[-1], final)
print(f"\nSaved: {final}")
display(Video(str(final), embed=True, width=512))

print("\nDownload:")
files.download(str(final))